In [1]:
import torch
from torch import nn

from model_utils import load_transformer_model, load_silence_latent, load_encoder, decode_latent_and_save_audio, \
    load_finetuning_audio_latents

C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\clip\clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_dtype = torch.bfloat16
model_repo = "ACE-Step/acestep-v15-turbo-shift1"

In [3]:
dit = load_transformer_model(model_repo, model_dtype, device)

C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:454: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:639: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\finite_scalar_quantization.py:159: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\lookup_free_quantization.py:244: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use 

In [4]:
for parameter in dit.parameters():
    parameter.requires_grad = False

In [5]:
rank = 8
for layer in dit.decoder.layers:
    layer.self_attn.q_proj.q_A = nn.Parameter(torch.randn(rank, layer.self_attn.q_proj.out_features, requires_grad=True, device=device, dtype=model_dtype))
    layer.self_attn.q_proj.q_B = nn.Parameter(torch.randn(layer.self_attn.q_proj.in_features, rank, requires_grad=True, device=device, dtype=model_dtype))
    layer.self_attn.v_proj.v_A = nn.Parameter(torch.randn(rank, layer.self_attn.v_proj.out_features, requires_grad=True, device=device, dtype=model_dtype))
    layer.self_attn.v_proj.v_B = nn.Parameter(torch.randn(layer.self_attn.v_proj.in_features, rank, requires_grad=True, device=device, dtype=model_dtype))


In [6]:
def lora_q_forward_hook(module, inputs, outputs):
    return outputs + (torch.dot(inputs, (module.q_B @ module.q_A).T))

def lora_v_forward_hook(module, inputs, outputs):
    return outputs + (torch.dot(inputs, (module.v_B @ module.v_A).T))

In [7]:
for layer in dit.decoder.layers:
    layer.self_attn.q_proj.register_forward_hook(lora_q_forward_hook)
    layer.self_attn.v_proj.register_forward_hook(lora_v_forward_hook)

In [8]:
vae = load_encoder("./models/ace-step-vae/config.json", "./models/ace-step-vae/checkpoint.ckpt", device, model_dtype)

C:\repos\audio-diffusion-finetuning\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [9]:
y = load_finetuning_audio_latents(vae, device, model_dtype)

torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([24, 2, 2880000])


RuntimeError: [enforce fail at alloc_cpu.cpp:117] data. DefaultCPUAllocator: not enough memory: you tried to allocate 17694720000 bytes.

In [ ]:
print(y.shape)

In [ ]:
# Generate music
silence_latent = load_silence_latent(model_repo, "silence_latent.pt", device, model_dtype)

text_hidden_states = torch.zeros(1, 77, 1024, dtype=model_dtype, device=device)
text_attention_mask = torch.zeros(text_hidden_states.shape[0], text_hidden_states.shape[1], dtype=torch.bool, device=device)
lyric_hidden_states = torch.zeros(1, 123, 1024, dtype=model_dtype, device=device)
lyric_attention_mask = torch.zeros(1, 123, dtype=torch.bool, device=device)

is_covers = torch.Tensor([False]).to(device)

seconds = 60
infer_steps = 50
frames_per_second = 25

seq_len = int(seconds * frames_per_second)

cur_chunk_mask = torch.ones(1, seq_len, 64, dtype=torch.bool, device=device)
cur_src_latents = silence_latent[:, :, :seq_len].permute(0, 2, 1)

outputs = dit.generate_audio(
    text_hidden_states=text_hidden_states,
    text_attention_mask=text_attention_mask,
    lyric_hidden_states=lyric_hidden_states,
    lyric_attention_mask=lyric_attention_mask,
    refer_audio_acoustic_hidden_states_packed=silence_latent,
    refer_audio_order_mask=torch.Tensor([0]).to(device),
    src_latents=cur_src_latents,
    chunk_masks=cur_chunk_mask,
    infer_steps=infer_steps,
    is_covers=is_covers,
    silence_latent=silence_latent,
    use_progress_bar=True,
    shift=1.0,

    repainting_start=torch.tensor([1.0]),
    repainting_end=torch.tensor([0.0]),
    audio_cover_strength=1.0,
    use_repainting=False
)
output_latents = outputs['target_latents'].transpose(1, 2).contiguous()
decode_latent_and_save_audio(output_latents, "lora.wav", vae)